# Week 1 — Task 2: Trends and the Spaghetti Problem

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

**Dataset:** `gapminder.csv` — 1,704 rows, 142 countries × 12 years (1952–2007).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('data/gapminder.csv')
print(f'Shape: {df.shape}')
print(f'Countries: {df["country"].nunique()}')
print(f'Years: {sorted(df["year"].unique())}')
print(f'Missing values:\n{df.isnull().sum()}')
df.head()

## Step 1 — The Messy Spaghetti Chart (Intentionally Bad)

Plot one line per country — all 142, with a legend.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
for country, grp in df.groupby('country'):
    ax.plot(grp['year'], grp['lifeExp'], lw=0.6, alpha=0.35, color='steelblue')

ax.set_xlabel('Year')
ax.set_ylabel('Life expectancy (years)')
ax.set_title('Life expectancy 1952-2007: 142 lines make individual comparison impossible [BAD EXAMPLE]')
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task2_spaghetti_bad.png', dpi=150, bbox_inches='tight')
plt.show()

### What is wrong with this chart:
1. **Overplotting** — 142 overlapping lines form a blue mass; individual countries are invisible.
2. **No legend is usable** — a 142-entry legend is unreadable and would dwarf the chart.
3. **No message** — the title only names the axes, not what the reader should take away.
4. **Colour carries no information** — every line is the same blue, so hue does nothing.
5. **The spaghetti obscures both trends** — neither the global rise nor individual outliers (e.g., Rwanda's collapse) can be seen.

## Step 2a — Fix: Aggregate (one line per continent)

In [ ]:
continent_colors = {
    'Africa':   '#d73027',
    'Americas': '#fc8d59',
    'Asia':     '#91bfdb',
    'Europe':   '#4575b4',
    'Oceania':  '#1a9850'
}

cont_agg = df.groupby(['continent', 'year'])['lifeExp'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
for cont, grp in cont_agg.groupby('continent'):
    ax.plot(grp['year'], grp['lifeExp'], lw=2.5, color=continent_colors[cont], label=cont)

ax.set_title('Average life expectancy rose in every continent — fastest in Asia', loc='left')
ax.set_xlabel('Year')
ax.set_ylabel('Life expectancy (years)')
ax.legend(title='Continent', frameon=False)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task2_continent_agg.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 2b — Fix: Highlight (grey all, colour 3 interesting countries)

In [ ]:
highlight = {
    'Rwanda':        '#d73027',   # collapsed during genocide, then recovered
    'China':         '#1a9850',   # massive sustained rise
    'United States': '#4575b4'    # slow but steady
}

fig, ax = plt.subplots(figsize=(12, 7))

for country, grp in df.groupby('country'):
    if country not in highlight:
        ax.plot(grp['year'], grp['lifeExp'], lw=0.5, alpha=0.2, color='#cccccc')

for country, color in highlight.items():
    grp = df[df['country'] == country].sort_values('year')
    ax.plot(grp['year'], grp['lifeExp'], lw=2.5, color=color)
    last = grp.iloc[-1]
    ax.text(last['year'] + 0.5, last['lifeExp'], country,
            color=color, va='center', fontsize=9, fontweight='bold')

ax.set_title("Rwanda's collapse and recovery stand out against a global rise (n=142 countries)", loc='left')
ax.set_xlabel('Year')
ax.set_ylabel('Life expectancy (years)')
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task2_highlight.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Comparing (a) vs (b)

**(a) Aggregate** shows continental-level trends clearly and supports direct comparison between continents. It completely hides within-continent variation — Rwanda's genocide trough disappears into the Africa mean.

**(b) Highlight** preserves individual country stories in their full context. You can see outliers and anomalies that aggregation hides. However, it cannot support comparison between continents — if you want to know whether Africa improved faster than Asia, (b) gives no clear answer.

**When to use which:** (a) is for structural regional comparisons; (b) is for storytelling about specific countries against the backdrop of the world.

## Step 4 — Has the Gap Between Richest and Poorest Narrowed Since 1952?

First, inspect the skewness of `gdpPercap`.

In [ ]:
year_data = df[df['year'] == 2007]['gdpPercap']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(year_data, bins=40, color='#2166ac', edgecolor='white')
axes[0].set_title('GDP per capita is extremely right-skewed (2007)', loc='left')
axes[0].set_xlabel('GDP per capita (USD)')
axes[0].set_ylabel('Number of countries')
for sp in ('top', 'right'):
    axes[0].spines[sp].set_visible(False)

axes[1].hist(np.log10(year_data), bins=30, color='#d73027', edgecolor='white')
axes[1].set_title('Log\u2081\u2080 scale reveals near-normal distribution (2007)', loc='left')
axes[1].set_xlabel('log\u2081\u2080(GDP per capita)')
axes[1].set_ylabel('Number of countries')
for sp in ('top', 'right'):
    axes[1].spines[sp].set_visible(False)

plt.tight_layout()
plt.savefig('task2_gdp_skew.png', dpi=150, bbox_inches='tight')
plt.show()
print('Decision: use log scale for GDP charts. Linear scale compresses poor countries into an unreadable band.')

In [ ]:
gap = df.groupby('year')['gdpPercap'].agg(
    p10=lambda x: np.percentile(x, 10),
    p90=lambda x: np.percentile(x, 90)
).reset_index()
gap['ratio'] = gap['p90'] / gap['p10']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: absolute incomes on log scale
axes[0].fill_between(gap['year'], gap['p10'], gap['p90'],
                     alpha=0.18, color='#4575b4')
axes[0].plot(gap['year'], gap['p90'], lw=2, color='#d73027', label='90th percentile (rich)')
axes[0].plot(gap['year'], gap['p10'], lw=2, color='#1a9850', label='10th percentile (poor)')
axes[0].set_yscale('log')
axes[0].set_title('Absolute gap widened — rich countries grew faster', loc='left')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('GDP per capita (log scale, USD)')
axes[0].legend(frameon=False)
for sp in ('top', 'right'):
    axes[0].spines[sp].set_visible(False)

# Panel 2: ratio of rich to poor
axes[1].plot(gap['year'], gap['ratio'], lw=2.5, color='#4575b4')
axes[1].axhline(gap['ratio'].iloc[0], color='#aaaaaa', lw=1, linestyle='--')
axes[1].set_title('P90/P10 income ratio rose from ~16x to ~26x — relative gap grew too', loc='left')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('P90/P10 GDP ratio')
axes[1].set_ylim(0, 35)
for sp in ('top', 'right'):
    axes[1].spines[sp].set_visible(False)

plt.tight_layout()
plt.savefig('task2_gap_chart.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart-type justification:** A dual-panel chart: (left) filled area on a log scale to show both the absolute income levels and the span between them; (right) ratio line chart to show the relative gap directly. Log scale is essential on the left because GDP is multiplicatively distributed — a 2× improvement at the bottom of the scale is economically equivalent to a 2× improvement at the top, and a linear axis hides this.

**Finding:** Neither the absolute gap nor the relative gap has narrowed. The richest 10% of countries grew faster in absolute dollars, and the P90/P10 ratio rose from roughly 16× in 1952 to roughly 26× by 2007. The global story is one of overall improvement *and* widening inequality.